<a href="https://colab.research.google.com/github/nathanchapero-creator/swapmeet_sales_analysis/blob/main/JewlerySalesCleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
!pip install gspread pandas

import gspread
import pandas as pd
import json
from google.colab import userdata

# 1. Pull the secret string from Colab's vault
creds_string = userdata.get('GOOGLE_CREDENTIALS')

# 2. Convert that string back into a JSON dictionary format
creds_dict = json.loads(creds_string)

# 3. Log in using the dictionary instead of a file!
gc = gspread.service_account_from_dict(creds_dict)

# Now extract your data exactly like before:
sheet = gc.open("Cypress Sales Transaction Log")
raw_tab = sheet.worksheet("Original Responses")
df_responses = pd.DataFrame(raw_tab.get_all_records())


In [49]:
# Pull Weather Data
weather_tab = sheet.worksheet("Weather Data")
df_weather = pd.DataFrame(weather_tab.get_all_records())

# Pull Gross Sales Data
gross_tab = sheet.worksheet("Gross Sales Data")
df_gross = pd.DataFrame(gross_tab.get_all_records())

In [50]:
# Normalizing Date Columns to create a derived key to join
df_responses["Merge Date"] = pd.to_datetime(df_responses["Timestamp"]).dt.normalize()
df_weather["Date"] = pd.to_datetime(df_weather["Date"])
df_gross["Date"] = pd.to_datetime(df_gross["Date"])

In [51]:
# Organizing Sterling Silver items
df_responses["Item Category"] = df_responses["Item Category"].replace({
    'Earring, Sterling Silver' : 'Sterling Silver Earring',
    'Ring, Sterling Silver' : 'Sterling Silver Ring',
    'Necklace, Sterling Silver' : 'Sterling Silver Necklace',
    'Necklace, Earring, Sterling Silver' : 'Sterling Silver Earring, Necklace'})

In [52]:
# Merging gross sales data with weather data ()
df_gross_weather = df_gross.merge(df_weather, on=["Date"], how = "left")

In [53]:
# Adding transaction IDs to each transaction in df_responses table
df_responses.insert(0, 'Transaction ID', "TXN-" + (df_responses.index + 1).astype(str))

In [54]:
# Copying responses table and splitting with delimiter comma (",")
df_responses_copy = df_responses.copy()
df_responses_copy['Item Category'] = df_responses_copy['Item Category'].str.split(pat=', ')

# Exploding "Item Category" to create separate item rows

df_responses_copy = df_responses_copy.explode('Item Category')

In [55]:
df_gross_weather.head(5)

,Date,Value,Notes,High Temp (F),Peak Condition,Morning Condition,Max Wind Speed (mph)
0,2023-12-24,926,,NaN,NaN,NaN,NaN
1,2023-12-30,40,Rain,NaN,NaN,NaN,NaN
2,2024-01-06,683,,NaN,NaN,NaN,NaN
3,2024-01-13,305,,NaN,NaN,NaN,NaN
4,2024-01-20,62,Rain,NaN,NaN,NaN,NaN


In [56]:
df_responses_copy.head(5)

,Transaction ID,Timestamp,Item Category,Price Range,Items Sold,Sold at Sticker Price?,Payment Method,Repeat Customer?,Merge Date
0,TXN-1,7/11/2026 8:08:19,Watch,$6 - $10,1,Yes (Full Price),Cash,No,2026-07-11
1,TXN-2,7/11/2026 8:48:58,Earring,$6 - $10,1,Yes (Full Price),Cash,No,2026-07-11
2,TXN-3,7/11/2026 8:49:31,Earring,$21 - $30,3,No (Negotiated Down),Cash,No,2026-07-11
3,TXN-4,7/11/2026 8:51:21,Clothes,$11 - $15,1,No (Negotiated Down),Cash,No,2026-07-11
4,TXN-5,7/11/2026 8:53:43,Earring,$21 - $30,2,No (Negotiated Down),Cash,Yes,2026-07-11


In [58]:
# 1. Connect to your newly created destination tabs
ws_items = sheet.worksheet("Clean Response Data")
ws_daily = sheet.worksheet("Clean Sales_Weather Data")

# 2. Clear the tabs to ensure a clean slate
ws_items.clear()
ws_daily.clear()

# 3. Create safe copies for the final export
df_responses_clean = df_responses_copy.copy()
df_gross_weather_clean = df_gross_weather.copy()

# 4. Convert Pandas Timestamps to plain strings so JSON can read them
df_responses_clean['Timestamp'] = df_responses_clean['Timestamp'].astype(str)
df_responses_clean['Merge Date'] = df_responses_clean['Merge Date'].astype(str)
df_gross_weather_clean['Date'] = df_gross_weather_clean['Date'].astype(str)

# 5. Replace any NaNs or NaTs (blanks) with empty strings
df_responses_clean = df_responses_clean.fillna('').replace('NaT', '')
df_gross_weather_clean = df_gross_weather_clean.fillna('').replace('NaT', '')

# 6. Export the DataFrames using USER_ENTERED to auto-parse dates
ws_items.update(
    values=[df_responses_clean.columns.values.tolist()] + df_responses_clean.values.tolist(),
    range_name='A1',
    value_input_option='USER_ENTERED'
)

ws_daily.update(
    values=[df_gross_weather_clean.columns.values.tolist()] + df_gross_weather_clean.values.tolist(),
    range_name='A1',
    value_input_option='USER_ENTERED'
)

# 7. Format the headers (Freeze, Bold, Filter)
ws_items.freeze(rows=1)
ws_daily.freeze(rows=1)
ws_items.format('1:1', {'textFormat': {'bold': True}})
ws_daily.format('1:1', {'textFormat': {'bold': True}})
ws_items.set_basic_filter()
ws_daily.set_basic_filter()

print("ETL Pipeline Complete! Data successfully pushed to Google Sheets.")

ETL Pipeline Complete! Data successfully pushed to Google Sheets.
